In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import folium
import geopandas as gpd
from shapely.geometry import Point

In [4]:
#Parsing Geojson file of ZIP Codes
gdf_ZIP_PA = gpd.read_file('../Spatial Joins/ACLED Join/pa_pennsylvania_zip_codes_geo.min.json')

In [6]:
#Checking CRS
gdf_ZIP_PA.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

#### Extracting adjacent ZIP Codes

In [8]:
#Creating new column to store bordering ZIP Codes
gdf_ZIP_PA['bordering_zips'] = None

for index, zip_polygon in gdf_ZIP_PA.iterrows():

    #Iterating over each polygon to define neighbours (i.e. ZIP Codes with touching borders)
    #Method: touches() returns TRUE/FALSE statements for all polygons that share a common border
    neighbors = gdf_ZIP_PA[gdf_ZIP_PA.geometry.touches(zip_polygon.geometry)]['ZCTA5CE10'].tolist()

    #filters out current ZIP code so only distinct neighbours are inlucded
    neighbors = [str(neighbor) for neighbor in neighbors if str(neighbor) != str(zip_polygon['ZCTA5CE10'])]
    gdf_ZIP_PA.at[index, 'bordering_zips'] = ', '.join(neighbors)

In [10]:
#print results
print(gdf_ZIP_PA[['ZCTA5CE10', 'bordering_zips']])

     ZCTA5CE10                                     bordering_zips
0        17301                  17362, 17316, 17364, 17350, 17331
1        17353                  17325, 17320, 17222, 17307, 17343
2        17325  17353, 17320, 17307, 17343, 17304, 17350, 1737...
3        15018           15642, 15037, 15135, 15089, 15028, 15083
4        15215                  15238, 15223, 15201, 15206, 15116
...        ...                                                ...
1793     17329                                17362, 17327, 17331
1794     17363                         17327, 17349, 17322, 17352
1795     17318                                17347, 17406, 17345
1796     17361                                17327, 17349, 17355
1797     17355                                17327, 17349, 17361

[1798 rows x 2 columns]


In [ ]:
#Convert to csv
gdf_ZIP_PA.to_csv("joined_ACLED_ZIP_22_PA.csv", index=False)
